# CourtView API Python Examples

This notebook demonstrates a full local workflow:

1. Health check against the API
2. Populate the database with **10 Anchorage criminal cases**
3. Include connected cases via defendant/co-defendant expansion
4. Use the Python example scripts and modules

No personal data is hardcoded in this notebook.

## Setup

Make sure the API is running (for example, `docker compose up -d`).

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

BASE_URL = os.getenv('COURTVIEW_API_BASE_URL', 'http://localhost:8088')
YEAR = datetime.now(timezone.utc).year
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_URL =', BASE_URL)
print('YEAR =', YEAR-1)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0:
        print(p.stderr)
        raise RuntimeError(f'command failed: {p.returncode}')
    return p

def run_json(cmd):
    p = run(cmd)
    return json.loads(p.stdout)

## 1) API health check via `courtview_api_client.py`

In [ ]:
health = run_json([
    sys.executable,
    'courtview_api_client.py',
    '--base-url', BASE_URL,
    'health',
])
health

## 2) Populate DB with 10 Anchorage criminal cases (+ connected defendant network)

Here we use the Python client module directly so we can set
`include_defendant_network=True`.

This is just to generate some information for testng purposes, populating the database sequentially would require a more robust system than this backfill.

In [ ]:
sys.path.insert(0, 'examples/python')
from courtview_api_client import CourtViewAPIClient

client = CourtViewAPIClient(base_url=BASE_URL, timeout_seconds=3000)

backfill = client.backfill_anchorage_criminal(
    count=10,
    year=YEAR,
    max_attempts=2000,
    timeout_seconds=7200,
    concurrency=2,
    include_defendant_network=True,
)

backfill_summary = backfill.get('summary', {})
case_numbers = backfill_summary.get('unique_case_numbers_captured', [])

print('Complete:', backfill_summary.get('complete'))
print('Found cases:', backfill_summary.get('found_cases'))
print('Persisted changed cases:', backfill_summary.get('persisted_changed_cases'))
print('Sample case numbers:', case_numbers[:5])

backfill_summary

## 3) Case-level criminal report via `criminal_charge_report.py`

This produces:
- non-dismissed charges
- charge state (`original`, `amended`, `downgraded`, `amended_downgraded`)
- conviction flag per case

In [ ]:
if not case_numbers:
    raise RuntimeError('No case numbers were captured from backfill')

sample_case = case_numbers[1]
sample_case

In [ ]:
report_json = OUTPUT_DIR / 'sample-criminal-report.json'
report_csv = OUTPUT_DIR / 'sample-criminal-report.csv'

summary = run_json([
    sys.executable,
    'criminal_charge_report.py',
    '--base-url', BASE_URL,
    '--case-number', sample_case,
    '--include-defendant-network',
    '--json-out', str(report_json),
    '--csv-out', str(report_csv),
    '--summary-only',
])

print('JSON report written to', report_json)
print('CSV report written to', report_csv)
summary

## 4) Runtime integration checks via `runtime_api_tests.py`

This discovers test subjects at runtime from backfilled cases and prints only redacted subject IDs.

In [ ]:
runtime_json = OUTPUT_DIR / 'runtime-tests.json'
runtime_result = run_json([
    sys.executable,
    'runtime_api_tests.py',
    '--base-url', BASE_URL,
    '--year', str(YEAR),
    '--backfill-count', '10',
    '--subject-count', '2',
    '--json-out', str(runtime_json),
])

print('Runtime test JSON written to', runtime_json)
runtime_result

## 5) Direct module usage (`courtview_criminal_analysis.py`)

This shows how to call the analysis layer directly from Python code.

In [ ]:
from courtview_criminal_analysis import build_criminal_defendant_report, flatten_report_rows

payload = client.search_case(
    case_number=sample_case,
    include_cases=True,
    include_defendant_network=True,
)

report = build_criminal_defendant_report(payload)
rows = flatten_report_rows(report)

print('records:', report.get('totals', {}).get('records'))
print('cases:', report.get('totals', {}).get('cases'))
print('rows:', len(rows))
rows[:3]

## Notes

- Generated outputs are written under `examples/python/output/` (gitignored).
- No personal data is hardcoded in this notebook.